In [1]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.messages import RemoveMessage

d:\LangGraph\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
model =  ChatGroq(
        model = "llama-3.1-8b-instant"
    )

In [4]:
def chat(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

def delete_old_messages(state: MessagesState):
    msgs = state["messages"]

    # if more than 10 messages, delete the earliest 6
    if len(msgs) > 10:
        to_remove = msgs[:6]
        return {"messages": [RemoveMessage(id=m.id) for m in to_remove]}

    return {}

In [5]:
builder = StateGraph(MessagesState)
builder.add_node("chat", chat)
builder.add_node("cleanup", delete_old_messages)

In [6]:
builder.add_edge(START, "chat")
builder.add_edge("chat", "cleanup")   # run deletion after each response
builder.add_edge("cleanup", "__end__")

In [7]:
graph = builder.compile(checkpointer=InMemorySaver())

In [8]:
config = {"configurable": {"thread_id": "t1"}}

In [9]:
# Run multiple turns
graph.invoke({"messages": [{"role": "user", "content": "Hi, I'm Amna"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "Tell me about LangGraph"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "Now explain checkpointers"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Langchain"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Quantum Mechanics"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Gen AI"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is my name"}]}, config)

{'messages': [HumanMessage(content='What is Langchain', additional_kwargs={}, response_metadata={}, id='7e323773-e828-4796-b506-55f98fe40c33'),
  AIMessage(content="**LangChain** is an open-source Python library that allows developers to build and interact with large language models (LLMs) in a more intuitive and flexible way. LangChain is designed to simplify the process of working with LLMs, making it easier to build custom applications and workflows.\n\nLangChain provides a set of tools and APIs that enable developers to:\n\n1. **Load and manage LLM models**: LangChain supports popular LLMs like LLaMA, BERT, and T5, making it easy to integrate them into your applications.\n2. **Create and manage chains**: A chain is a sequence of tasks that are executed in a specific order. LangChain allows you to create and manage chains, enabling you to build complex workflows and interact with LLMs in a more flexible way.\n3. **Perform tasks and operations**: LangChain provides a range of APIs fo

In [10]:
snap = graph.get_state(config)
print("Stored messages after cleanup:", len(snap.values["messages"]))

Stored messages after cleanup: 8
